# BTC/USDT Data Exploration

This notebook explores historical Bitcoin/USDT data to prepare for training a reinforcement learning trading agent.

In [11]:
import pandas as pd
import pickle
from pathlib import Path

import plotly.graph_objects as go
import plotly.express as px

## Loading the data
Loading historical BTC/USDT data in pickle format (hourly OHLCV data).

In [12]:
raw_data_path = Path('../data/raw/binance-BTCUSDT-1h.pkl')

with open(raw_data_path, 'rb') as f:
    df = pickle.load(f)

print(f"shape: {df.shape}")
display(df.head(10))

shape: (44967, 6)


,open,high,low,close,volume,date_close
date_open,,,,,,
2020-09-30 23:00:00,10745.85,10785.00,10735.51,10776.59,1235.545956,2020-10-01 00:00:00
2020-10-01 00:00:00,10776.59,10826.19,10776.59,10788.06,2128.759531,2020-10-01 01:00:00
2020-10-01 01:00:00,10788.30,10849.97,10786.74,10838.88,1604.129560,2020-10-01 02:00:00
2020-10-01 02:00:00,10838.89,10857.47,10807.39,10817.14,1268.291734,2020-10-01 03:00:00
2020-10-01 03:00:00,10817.14,10824.22,10789.01,10798.18,939.599057,2020-10-01 04:00:00
2020-10-01 04:00:00,10798.38,10826.42,10795.78,10800.01,1082.441958,2020-10-01 05:00:00
2020-10-01 05:00:00,10800.00,10826.00,10790.85,10821.07,1264.550975,2020-10-01 06:00:00
2020-10-01 06:00:00,10821.07,10844.71,10811.72,10821.29,1746.032168,2020-10-01 07:00:00
2020-10-01 07:00:00,10821.29,10839.92,10799.70,10824.75,1169.162524,2020-10-01 08:00:00


## Price Visualization

Candlestick chart to visualize the evolution of BTC/USDT price over the entire available period.


In [13]:
increasing_color = '#2ecc40'  
decreasing_color = '#ff4136' 
dark_blue = '#001f3f'

fig = go.Figure(
    data=[
        go.Candlestick(
            x=df.index,
            open=df['open'],
            high=df['high'],
            low=df['low'],
            close=df['close'],
            name="BTCUSDT",
            increasing=dict(line=dict(color=increasing_color), fillcolor=increasing_color),
            decreasing=dict(line=dict(color=decreasing_color), fillcolor=decreasing_color),
        )
    ]
)
fig.update_layout(
    title="BTC/USDT 1h",
    xaxis_title="Date",
    yaxis_title="Price",
    plot_bgcolor=dark_blue,
    paper_bgcolor=dark_blue,
    font_color='white',
    xaxis=dict(color='white'),
    yaxis=dict(color='white'),
)
fig.show()


## Train/Evaluation Split

The dataset is divided into two periods:
- **Training period**: from 2024-10-01 to 2025-09-30
- **Evaluation period**: from 2025-09-30 to 2025-11-01

The plot below shows the closing price evolution for each period, along with variation statistics.


In [14]:
train_start = "2024-10-01"
train_end = "2025-09-30"
eval_start = "2025-10-01"
eval_end = "2025-11-01"

df_train = df[(df.index >= train_start) & (df.index <= train_end)]
df_eval = df[(df.index >= eval_start) & (df.index <= eval_end)]

# Compute variations for each period (absolute and percentage change)
def compute_variations(df_period):
    if df_period.empty:
        return None, None, None, None
    close_start = df_period["close"].iloc[0]
    close_end = df_period["close"].iloc[-1]
    abs_change = close_end - close_start
    pct_change = (abs_change / close_start) * 100
    return close_start, close_end, abs_change, pct_change

train_start_price, train_end_price, train_abs_change, train_pct_change = compute_variations(df_train)
eval_start_price, eval_end_price, eval_abs_change, eval_pct_change = compute_variations(df_eval)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_train.index,
    y=df_train["close"],
    mode="lines",
    name="Training period",
    line=dict(color="#0000cc"),
))

fig.add_trace(go.Scatter(
    x=df_eval.index,
    y=df_eval["close"],
    mode="lines",
    name="Evaluation period",
    line=dict(color="#ffaa3c"),
))

# Add annotations for each period containing key info
if train_start_price is not None:
    train_text = (
        f"Training period<br>"
        f"Start: {train_start_price:.2f}<br>"
        f"End: {train_end_price:.2f}<br>"
        f"Δ: {train_abs_change:+.2f} ({train_pct_change:+.2f}%)"
    )
    fig.add_annotation(
        x=df_train.index[int(len(df_train)//2)] if len(df_train) else train_start,
        y=max(df_train["close"].max(), df_eval["close"].max()),
        text=train_text,
        showarrow=False,
        align="left",
        bordercolor="#0000cc",
        borderwidth=2,
        borderpad=6,
        bgcolor="white",
        font=dict(color="#0000cc"),
        xanchor="center",
        yanchor="top"
    )

if eval_start_price is not None:
    eval_text = (
        f"Evaluation period<br>"
        f"Start: {eval_start_price:.2f}<br>"
        f"End: {eval_end_price:.2f}<br>"
        f"Δ: {eval_abs_change:+.2f} ({eval_pct_change:+.2f}%)"
    )
    fig.add_annotation(
        x=df_eval.index[int(len(df_eval)//2)] if len(df_eval) else eval_start,
        y=min(df_train["close"].min(), df_eval["close"].min()),
        text=eval_text,
        showarrow=False,
        align="left",
        bordercolor="#ffaa3c",
        borderwidth=2,
        borderpad=6,
        bgcolor="white",
        font=dict(color="#ffaa3c"),
        xanchor="center",
        yanchor="bottom"
    )

fig.update_layout(
    title="Close price BTC/USDT<br><span style='font-size:13px'>Training and Evaluation Periods</span>",
    xaxis_title="Date",
    yaxis_title="Close Price",
    legend=dict(
        x=0.01, 
        y=0.99, 
        bgcolor="rgba(0,0,0,0)",
        font=dict(color="black")
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    font_color='black',
    xaxis=dict(color='black'),
    yaxis=dict(color='black'),
)
fig.show()


In [15]:
#df.to_csv('../data/processed/data_v1.csv', index=True)

## Additional columns


In [16]:
window = 10 

df['momentum'] = df['close'].diff().fillna(0)
df['intraday_vol'] = df['high'] - df['low']
df['ma'] = df['close'].rolling(window=window).mean()

display(df.head(10))

,open,high,low,close,volume,date_close,momentum,intraday_vol,ma
date_open,,,,,,,,,
2020-09-30 23:00:00,10745.85,10785.00,10735.51,10776.59,1235.545956,2020-10-01 00:00:00,0.00,49.49,NaN
2020-10-01 00:00:00,10776.59,10826.19,10776.59,10788.06,2128.759531,2020-10-01 01:00:00,11.47,49.60,NaN
2020-10-01 01:00:00,10788.30,10849.97,10786.74,10838.88,1604.129560,2020-10-01 02:00:00,50.82,63.23,NaN
2020-10-01 02:00:00,10838.89,10857.47,10807.39,10817.14,1268.291734,2020-10-01 03:00:00,-21.74,50.08,NaN
2020-10-01 03:00:00,10817.14,10824.22,10789.01,10798.18,939.599057,2020-10-01 04:00:00,-18.96,35.21,NaN
2020-10-01 04:00:00,10798.38,10826.42,10795.78,10800.01,1082.441958,2020-10-01 05:00:00,1.83,30.64,NaN
2020-10-01 05:00:00,10800.00,10826.00,10790.85,10821.07,1264.550975,2020-10-01 06:00:00,21.06,35.15,NaN
2020-10-01 06:00:00,10821.07,10844.71,10811.72,10821.29,1746.032168,2020-10-01 07:00:00,0.22,32.99,NaN
2020-10-01 07:00:00,10821.29,10839.92,10799.70,10824.75,1169.162524,2020-10-01 08:00:00,3.46,40.22,NaN


In [17]:
df.to_csv('../data/processed/data_v2.csv', index=True)

In [18]:
# Calculate MACD (Moving Average Convergence Divergence) with dynamic spans
macd_short_span = 12
macd_long_span = 26

exp_short = df['close'].ewm(span=macd_short_span, adjust=False).mean()
exp_long = df['close'].ewm(span=macd_long_span, adjust=False).mean()
df['macd'] = exp_short - exp_long

rsi_window = 14
delta = df['close'].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(window=rsi_window, min_periods=rsi_window).mean()
avg_loss = loss.rolling(window=rsi_window, min_periods=rsi_window).mean()

small_value = 1e-10  # To avoid division by zero
rs = avg_gain / (avg_loss + small_value)
df['rsi'] = 100 - (100 / (1 + rs))

display(df.head(10))
#df.to_csv('../data/processed/data_v3.csv', index=True)


,open,high,low,close,volume,date_close,momentum,intraday_vol,ma,macd,rsi
date_open,,,,,,,,,,,
2020-09-30 23:00:00,10745.85,10785.00,10735.51,10776.59,1235.545956,2020-10-01 00:00:00,0.00,49.49,NaN,0.000000,NaN
2020-10-01 00:00:00,10776.59,10826.19,10776.59,10788.06,2128.759531,2020-10-01 01:00:00,11.47,49.60,NaN,0.914986,NaN
2020-10-01 01:00:00,10788.30,10849.97,10786.74,10838.88,1604.129560,2020-10-01 02:00:00,50.82,63.23,NaN,5.675445,NaN
2020-10-01 02:00:00,10838.89,10857.47,10807.39,10817.14,1268.291734,2020-10-01 03:00:00,-21.74,50.08,NaN,7.606227,NaN
2020-10-01 03:00:00,10817.14,10824.22,10789.01,10798.18,939.599057,2020-10-01 04:00:00,-18.96,35.21,NaN,7.519788,NaN
2020-10-01 04:00:00,10798.38,10826.42,10795.78,10800.01,1082.441958,2020-10-01 05:00:00,1.83,30.64,NaN,7.512353,NaN
2020-10-01 05:00:00,10800.00,10826.00,10790.85,10821.07,1264.550975,2020-10-01 06:00:00,21.06,35.15,NaN,9.100917,NaN
2020-10-01 06:00:00,10821.07,10844.71,10811.72,10821.29,1746.032168,2020-10-01 07:00:00,0.22,32.99,NaN,10.259354,NaN
2020-10-01 07:00:00,10821.29,10839.92,10799.70,10824.75,1169.162524,2020-10-01 08:00:00,3.46,40.22,NaN,11.326057,NaN


In [19]:
window = 120 


df['ma'] = df['close'].rolling(window=window).mean()
df.head()


,open,high,low,close,volume,date_close,momentum,intraday_vol,ma,macd,rsi
date_open,,,,,,,,,,,
2020-09-30 23:00:00,10745.85,10785.00,10735.51,10776.59,1235.545956,2020-10-01 00:00:00,0.00,49.49,NaN,0.000000,NaN
2020-10-01 00:00:00,10776.59,10826.19,10776.59,10788.06,2128.759531,2020-10-01 01:00:00,11.47,49.60,NaN,0.914986,NaN
2020-10-01 01:00:00,10788.30,10849.97,10786.74,10838.88,1604.129560,2020-10-01 02:00:00,50.82,63.23,NaN,5.675445,NaN
2020-10-01 02:00:00,10838.89,10857.47,10807.39,10817.14,1268.291734,2020-10-01 03:00:00,-21.74,50.08,NaN,7.606227,NaN
2020-10-01 03:00:00,10817.14,10824.22,10789.01,10798.18,939.599057,2020-10-01 04:00:00,-18.96,35.21,NaN,7.519788,NaN


In [20]:
df.to_csv('../data/processed/data_v4.csv', index=True)